<h1>Imports</h1>

In [1]:
import shutil
import subprocess
import sys
from itertools import product
from pathlib import Path



In [2]:
def _ensure_installed(pkg_import_name, pip_name=None):
    try:
        __import__(pkg_import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pip_name or pkg_import_name]
        )

_ensure_installed("huggingface_hub")


In [3]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt 
from PIL import Image
from torchvision import transforms 
from torchvision.utils import save_image
from huggingface_hub import hf_hub_download

In [4]:
Device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device:{Device}")

Using Device:cuda


<h1>Model Definition</h1>

In [ ]:
vgg = nn.Sequential(
    nn.Conv2d(3, 3, (1, 1)),                        
    nn.Conv2d(3, 64, (3, 3), padding=1),            
    nn.ReLU(),                                       
    nn.Conv2d(64, 64, (3, 3), padding=1),           
    nn.ReLU(),                                       
    nn.MaxPool2d((2, 2), (2, 2), (0, 0), ceil_mode=True),  
    nn.Conv2d(64, 128, (3, 3), padding=1),          
    nn.ReLU(),                                       
    nn.Conv2d(128, 128, (3, 3), padding=1),         
    nn.ReLU(),                                     
    nn.MaxPool2d((2, 2), (2, 2), (0, 0), ceil_mode=True),  
    nn.Conv2d(128, 256, (3, 3), padding=1),         
    nn.ReLU(),                                      
    nn.Conv2d(256, 256, (3, 3), padding=1),         
    nn.ReLU(),                                     
    nn.Conv2d(256, 256, (3, 3), padding=1),         
    nn.ReLU(),                                      
    nn.Conv2d(256, 256, (3, 3), padding=1),       
    nn.ReLU(),                                       
    nn.MaxPool2d((2, 2), (2, 2), (0, 0), ceil_mode=True),  
    nn.Conv2d(256, 512, (3, 3), padding=1),         
    nn.ReLU(),                                       
)

In [ ]:
decoder = nn.Sequential(
    nn.Conv2d(512, 256, (3, 3), padding=1),
    nn.ReLU(),
    nn.Upsample(scale_factor=2, mode="nearest"),
    nn.Conv2d(256, 256, (3, 3), padding=1),
    nn.ReLU(),
    nn.Conv2d(256, 256, (3, 3), padding=1),
    nn.ReLU(),
    nn.Conv2d(256, 256, (3, 3), padding=1),
    nn.ReLU(),
    nn.Conv2d(256, 128, (3, 3), padding=1),
    nn.ReLU(),
    nn.Upsample(scale_factor=2, mode="nearest"),
    nn.Conv2d(128, 128, (3, 3), padding=1),
    nn.ReLU(),
    nn.Conv2d(128, 64, (3, 3), padding=1),
    nn.ReLU(),
    nn.Upsample(scale_factor=2, mode="nearest"),
    nn.Conv2d(64, 64, (3, 3), padding=1),
    nn.ReLU(),
    nn.Conv2d(64, 3, (3, 3), padding=1),
)


<h1>Calculation for Adaptive INstance based Normalization</h1>

In [ ]:
# defining the mean calcualation fucntions 
def cal_mean_std(feat,eps=1e-5):
    size = feat.size()
    assert len(size)==4, "expected (N, C, H, W)"
    N,C = size[:2]
    feat_flat=feat.view(N, C, -1)
    feat_mean=feat_flat.mean(dim=2,keepdim=True)
    feat_var=((feat_flat-feat_mean)**2).mean(dim=2,keepdim=True) + eps
    feat_std = feat_var.sqrt()
    return feat_mean.view(N,C,1,1),feat_std.view(N,C,1,1)

In [ ]:
def adaptive_instance_normalization(content_feat, style_feat):
    assert content_feat.size()[:2]==style_feat.size()[:2]
    content_mean,content_std=calc_mean_std(content_feat)
    style_mean,style_std=calc_mean_std(style_feat)
    normalized_feat=(content_feat-content_mean)/content_std
    return normalized_feat*style_std+style_mean
    

In [ ]:
    
def style_transfer(vgg_enc, dec, content, style, alpha=1.0):
    assert 0.0<=alpha<=1.0 "aplha between the 0 and 1"
    content_f=vgg_enc(content)
    style_f=vgg_enc(style)
    feat=adaptive_instance_normalization(content_f,style_f)
    feat=feat*alpha + content_f*(1-alpha)
    return dec(feat)
    

In [ ]:
def style_transfer_interpolate(vgg_enc,dec,content,styles,weights,alpha=1.0):
    assert len(styles)==len(weights)
    assert abs(sum(weights)-1.0)<1e-3, "INTERP_WEIGHTS should sum to 1.0"
    content_f=vgg_enc(content)
    feat=torch.zeros_like(content_f)
    for style,w in zip(styles,weights):
        style_f=vgg_enc(style)
        feat=feat+w*adaptive_instance_normalization(content_f,style_f)
    feat= feat*alpha+content_f*(1-alpha)
    return dec(feat)
    

In [ ]:
def style_transfer_spatial(vgg_enc, dec, content, styles, masks, alpha=1.0):
    assert len(styles)==len(masks)
    content_f=vgg_enc(content)
    _, _, fh, fw=content_f.shape
    feat=torch.zeros_like(content_f)
    mask_sum = torch.zeros(1, 1,fh,fw, device=content_f.device)
    for style, mask in zip(styles,masks):
        style_f=vgg_enc(style)
        adain_feat=adaptive_instance_normalization(content_f, style_f)
        mask_r=F.interpolate(mask, size=(fh, fw), mode="nearest")
        feat=feat + adain_feat * mask_r
        mask_sum=mask_sum + mask_r
    mask_sum=mask_sum.clamp(min=1e-5)
    feat=feat/mask_sum
    feat=feat*alpha+content_f*(1- alpha)
    return dec(feat)

<h1>Color preservation</h1>

In [ ]:
#  i wasnt understanding this but needed so i used AI here 
def _flatten_mean_std(img):
    flat=img.reshape(3, -1)
    mean=flat.mean(dim=-1,keepdim=True)
    std=flat.std(dim=-1,keepdim=True)+1e-5
    return flat,mean,std

In [ ]:
def _mat_sqrt(x):
    U,S,Vh=torch.linalg.svd(x)
    return U @ torch.diag(S.clamp(min=0).sqrt()) @ Vh

In [ ]:
def coral(source, target):
    source_flat,source_mean,source_std=_flatten_mean_std(source)
    source_norm=(source_flat-source_mean)/source_std
    source_cov_eye=source_norm @ source_norm.t()+torch.eye(3,device=source.device)
    target_flat,target_mean,target_std=_flatten_mean_std(target)
    target_norm=(target_flat-target_mean)/target_std
    target_cov_eye=target_norm @ target_norm.t() + torch.eye(3, device=target.device)
    transferred_norm=_mat_sqrt(target_cov_eye) @ torch.inverse(_mat_sqrt(source_cov_eye)) @ source_norm
    transferred=transferred_norm * target_std + target_mean
    return transferred.reshape(source.size()).clamp(0, 1)

In [ ]:
def maybe_preserve_color(style_batched, content_batched, preserve_color):
    if not preserve_color:
        return style_batched
    recolored = coral(style_batched.squeeze(0), content_batched.squeeze(0))
    return recolored.unsqueeze(0)

<h1>Weight download & model loading</h1>

In [ ]:
def download_weights(models_dir):
    models_dir=Path(models_dir)
    models_dir.mkdir(exist_ok=True,parents=True)
    repo_id="tidalove/adain"
    paths={}
    for fname in ["vgg_normalized.pth","decoder.pth"]:
        dest=models_dir/fname
        if dest.is_file():
            print(f"Found cached{fname}")
        else:
            print(f"Downloading {fname} ...")
            cached=hf_hub_download(repo_id=repo_id,filename=fname,repo_type="space")
            shutil.copy(cached,dest)
            print(f"Saved to {dest}")
        paths[fname]=str(dest)
    return paths

In [ ]:
def load_state_dict_report(module, state_dict, name):
    result=module.load_state_dict(state_dict,strict=False)
    missing,unexpected=result.missing_keys,result.unexpected_keys
    if unexpected:
        print(f"[{name}] ignored {len(unexpected)} checkpoint key(s) not used by this module (expected)")
    if missing:
        print(f"[{name}] MISSING {len(missing)} key(s) this module needs: {missing}")
    assert not missing, (
        f"[{name}] refusing to continue — {len(missing)} required parameter(s) were not "
    )
    print(f"[{name}] loaded OK ({sum(p.numel() for p in module.parameters())} params)")

In [ ]:
def load_models(models_dir):
    weight_paths=download_weights(models_dir)
    vgg_state=torch.load(weight_paths["vgg_normalized.pth"],map_location=DEVICE)
    decoder_state=torch.load(weight_paths["decoder.pth"],map_location=DEVICE)
    load_state_dict_report(vgg,vgg_state,"encoder")
    load_state_dict_report(decoder,decoder_state,"decoder")
    vgg.eval().to(DEVICE)
    decoder.eval().to(DEVICE)
    for p in vgg.parameters():
        p.requires_grad_(False)
    for p in decoder.parameters():
        p.requires_grad_(False)
    return vgg, decoder

<h1>Io helpers</h1>

In [ ]:
def make_transform(size):
    tf_list=[]
    if size!= 0:
        tf_list.append(transforms.Resize(size))
    tf_list.append(transforms.ToTensor())
    return transforms.Compose(tf_list)

In [ ]:
def load_image(path,size):
    tf=make_transform(size)
    img=Image.open(path).convert("RGB")
    return tf(img).unsqueeze(0).to(DEVICE)

In [ ]:
def load_mask(path, hw):
    tf=transforms.Compose([transforms.Resize(hw),transforms.ToTensor()])
    img=Image.open(path).convert("L")
    return tf(img).unsqueeze(0).to(DEVICE)

In [ ]:
def tensor_to_display(t):
    return t.clamp(0, 1).squeeze(0).permute(1,2,0).cpu().numpy()

In [ ]:
def show_row(images, titles, suptitle=None):
    fig,axes = plt.subplots(1,len(images),figsize=(5*len(images),5))
    if len(images)==1:
        axes=[axes]
    for ax,img,title in zip(axes,images,titles):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
    if suptitle:
        fig.suptitle(suptitle)
    plt.tight_layout()
    plt.show()

<h1>Mode runners</h1>

In [5]:
def run_single(vgg_enc, dec, out_dir, cfg):
    saved = []
    for content_path,style_path,alpha in product(cfg["CONTENT_IMAGES"], cfg["STYLE_IMAGES"], cfg["ALPHAS"]):
        if not Path(content_path).is_file():
            print(f"Skipping: content file not found: {content_path}")
            continue
        if not Path(style_path).is_file():
            print(f"Skipping: style file not found: {style_path}")
            continue

        
        content=load_image(content_path,cfg["CONTENT_SIZE"])
        style=load_image(style_path,cfg["STYLE_SIZE"])
        style=maybe_preserve_color(style,content,cfg["PRESERVE_COLOR"])

        with torch.no_grad():
            output = style_transfer(vgg_enc,dec,content,style,alpha)

        out_path = out_dir / f"{Path(content_path).stem}_stylized_{Path(style_path).stem}_alpha{alpha}.jpg"
        save_image(output.cpu().clamp(0, 1), str(out_path))
        saved.append(out_path)
        print(f"Saved: {out_path}")

        show_row(
            [tensor_to_display(content), tensor_to_display(style), tensor_to_display(output)],
            ["Content", "Style" + (" (color-matched)" if cfg["PRESERVE_COLOR"] else ""), f"Result (alpha={alpha})"],
        )
    return saved

In [6]:
def run_interpolate(vgg_enc, dec, out_dir, cfg):
    if not Path(cfg["INTERP_CONTENT"]).is_file():
        sys.exit(f"Content file not found: {cfg['INTERP_CONTENT']}")
        
    for p in cfg["INTERP_STYLES"]:
        if not Path(p).is_file():
            sys.exit(f"Style file not found: {p}")

    content = load_image(cfg["INTERP_CONTENT"],cfg["CONTENT_SIZE"])
    styles = [load_image(p, cfg["STYLE_SIZE"]) for p in cfg["INTERP_STYLES"]]
    styles = [maybe_preserve_color(s, content,cfg["PRESERVE_COLOR"]) for s in styles]

    with torch.no_grad():
        output=style_transfer_interpolate(vgg_enc,dec,content,styles,cfg["INTERP_WEIGHTS"],cfg["INTERP_ALPHA"])
    # this cpde is written by ai
    style_stems = "_".join(Path(p).stem for p in cfg["INTERP_STYLES"])
    out_path = out_dir / f"{Path(cfg['INTERP_CONTENT']).stem}_interp_{style_stems}_alpha{cfg['INTERP_ALPHA']}.jpg"
    save_image(output.cpu().clamp(0, 1), str(out_path))
    print(f"Saved: {out_path}")

    images = [tensor_to_display(content)] + [tensor_to_display(s) for s in styles] + [tensor_to_display(output)]
    titles = ["Content"] + [f"Style {i+1} (w={w})" for i, w in enumerate(cfg["INTERP_WEIGHTS"])] + ["Blended result"]
    show_row(images, titles)
    return [out_path]

In [ ]:
# this code is written by ai
def run_spatial(vgg_enc, dec, out_dir, cfg):
    if not Path(cfg["SPATIAL_CONTENT"]).is_file():
        sys.exit(f"Content file not found: {cfg['SPATIAL_CONTENT']}")
    for p in cfg["SPATIAL_STYLES"] + cfg["SPATIAL_MASKS"]:
        if not Path(p).is_file():
            sys.exit(f"File not found: {p}")
    if len(cfg["SPATIAL_STYLES"]) != len(cfg["SPATIAL_MASKS"]):
        sys.exit("SPATIAL_STYLES and SPATIAL_MASKS must be the same length")

    content = load_image(cfg["SPATIAL_CONTENT"], cfg["CONTENT_SIZE"])
    hw = content.shape[-2:]
    styles = [load_image(p, cfg["STYLE_SIZE"]) for p in cfg["SPATIAL_STYLES"]]
    styles = [maybe_preserve_color(s, content, cfg["PRESERVE_COLOR"]) for s in styles]
    masks = [load_mask(p, hw) for p in cfg["SPATIAL_MASKS"]]

    with torch.no_grad():
        output = style_transfer_spatial(vgg_enc, dec, content, styles, masks, cfg["SPATIAL_ALPHA"])

    style_stems = "_".join(Path(p).stem for p in cfg["SPATIAL_STYLES"])
    out_path = out_dir / f"{Path(cfg['SPATIAL_CONTENT']).stem}_spatial_{style_stems}.jpg"
    save_image(output.cpu().clamp(0, 1), str(out_path))
    print(f"Saved: {out_path}")

    images = [tensor_to_display(content)]
    titles = ["Content"]
    for s, m in zip(styles, masks):
        images += [tensor_to_display(s), m.squeeze().cpu().numpy()]
        titles += ["Style", "Mask"]
    images.append(tensor_to_display(output))
    titles.append("Spatial result")
    show_row(images, titles)
    return [out_path]

<h1>Load models</h1>

In [7]:
MODELS_DIR = "/kaggle/working/models"
vgg_enc, dec = load_models(MODELS_DIR)
print("Models loaded.")

NameError: name 'load_models' is not defined

In [ ]:
def inspect_checkpoint(path):
    sd = torch.load(path,map_location="cpu")
    if hasattr(sd, "state_dict"):  
        sd = sd.state_dict()
    for k, v in sd.items():
        print(f"{k:20s} {tuple(v.shape)}")

In [8]:
CONFIG = {
    # Which mode to run:
    #   "single"      — one or more content images x one or more style images
    #                    (one output per pair)
    #   "interpolate" — one content image, blended across MULTIPLE styles at
    #                    once into a single output (Eq. 15)
    #   "spatial"     — one content image, different styles applied to
    #                    different masked regions of the SAME output (Fig. 10)
    "MODE": "single",

    # Apply color preservation (Fig. 9) in ALL modes: recolors each style
    # image to match the content image's color distribution before
    # extracting style features, so brushstroke/texture transfers but
    # original colors stay.
    "PRESERVE_COLOR": False,

    # Resize the shorter side to this many pixels before processing.
    # Set to 0 to keep original size (slower / more memory, especially on CPU).
    "CONTENT_SIZE": 512,
    "STYLE_SIZE": 512,

    "OUTPUT_DIR": "/kaggle/working/output",

    # --- "single" mode settings ---
    "CONTENT_IMAGES": [
        "/kaggle/input/datasets/roshanbhatta567/content/WhatsApp Image 2026-05-16 at 11.48.09.jpeg",
    ],
    "STYLE_IMAGES": [
        "/kaggle/input/datasets/roshanbhatta567/style2/ghibli.png",
    ],
    # Stylization strength(s) to try. 0.0 = untouched photo, 1.0 = full style.
    "ALPHAS": [0.3],

    # --- "interpolate" mode settings ---
    "INTERP_CONTENT": "/kaggle/working/your_photo.jpg",
    "INTERP_STYLES": [
        "/kaggle/working/style1.jpg",
        "/kaggle/working/style2.jpg",
    ],
    # Must be same length as INTERP_STYLES and sum to 1.0 (e.g. [0.5, 0.5])
    "INTERP_WEIGHTS": [0.5, 0.5],
    "INTERP_ALPHA": 1.0,

    # --- "spatial" mode settings ---
    "SPATIAL_CONTENT": "/kaggle/working/your_photo.jpg",
    "SPATIAL_STYLES": [
        "/kaggle/working/style1.jpg",
        "/kaggle/working/style2.jpg",
    ],
    # One grayscale mask per style, same length/order as SPATIAL_STYLES.
    # White (255) = "apply this style here", black (0) = "not here". Masks
    # are resized to match the content image automatically.
    "SPATIAL_MASKS": [
        "/kaggle/working/mask1.png",
        "/kaggle/working/mask2.png",
    ],
    "SPATIAL_ALPHA": 1.0,
}


In [ ]:
out_dir = Path(CONFIG["OUTPUT_DIR"])
out_dir.mkdir(exist_ok=True, parents=True)

mode = CONFIG["MODE"]
if mode == "single":
    saved = run_single(vgg_enc, dec, out_dir, CONFIG)
elif mode == "interpolate":
    saved = run_interpolate(vgg_enc, dec, out_dir, CONFIG)
elif mode == "spatial":
    saved = run_spatial(vgg_enc, dec, out_dir, CONFIG)
else:
    sys.exit(f'Unknown MODE "{mode}". Use "single", "interpolate", or "spatial".')

if saved:
    print(f"\nDone. {len(saved)} image(s) saved under {out_dir}/")
    print("On Kaggle, anything in /kaggle/working/ is automatically saved")
    print("as notebook output and downloadable from the Output tab.")
else:
    print("\nNo images were processed — check the file paths in CONFIG above.")
